# Notebook 02 - Adapt Dataset With Adaption

This notebook follows the Adaption SDK flow from the docs: install/import the SDK, load credentials, upload the prepared file, confirm column mapping, run an estimate, run a small pilot, poll for completion, download the adapted output, and inspect before/after rows.

API cells are guarded by `RUN_ADAPTION_API = False` by default.

In [ ]:
# If your environment does not have the SDK yet, uncomment this line:
# %pip install adaption

from pathlib import Path
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
sys.path.append(str(ROOT))

from kirundi_sft_starter.utils import load_env, load_yaml
from scripts.adapt_with_adaption import blueprint_text

load_env()
run_cfg = load_yaml(ROOT / "configs/adaption_run.yaml")
blueprint_cfg = load_yaml(ROOT / "configs/adaption_blueprint.yaml")

RUN_ADAPTION_API = False
HAS_KEY = bool(os.environ.get("ADAPTION_API_KEY"))
print("Adaption key loaded:", HAS_KEY)


## Inspect the input and column mapping

Adaption needs to know which columns contain the prompt and completion. This repo maps `instruction` to `prompt` and `response` to `completion`.

In [ ]:
input_path = ROOT / run_cfg["input_path"]
df = pd.read_csv(input_path)
display(df.head())
run_cfg["column_mapping"]

In [ ]:
brand_controls = dict(blueprint_cfg["adaption_brand_controls"])
brand_controls["blueprint"] = blueprint_text(blueprint_cfg)

print(brand_controls["blueprint"][:1200])

## Run estimate and pilot

Leave `RUN_ADAPTION_API = False` until you have reviewed the input, column mapping, and blueprint. The CLI version below also supports `--dry-run`.

In [ ]:
if RUN_ADAPTION_API and HAS_KEY:
    from adaption import Adaption
    client = Adaption(api_key=os.environ["ADAPTION_API_KEY"])
    upload = client.datasets.upload_file(str(input_path), name=run_cfg["dataset_name"])
    dataset_id = upload.dataset_id
    print("Dataset ID:", dataset_id)

    estimate = client.datasets.run(
        dataset_id,
        column_mapping=run_cfg["column_mapping"],
        brand_controls=brand_controls,
        recipe_specification=blueprint_cfg["adaption_recipe_specification"],
        job_specification={"max_rows": run_cfg["pilot"]["max_rows"]},
        estimate=True,
    )
    print("Estimated credits:", estimate.estimated_credits_consumed)
else:
    print("Dry run. Review configs/adaption_run.yaml, then run the CLI command below when ready.")

## CLI Equivalent

```bash
python scripts/adapt_with_adaption.py --dry-run
python scripts/adapt_with_adaption.py
# Optional full run after the pilot looks good:
python scripts/adapt_with_adaption.py --full
```

After download, convert the adapted output into the same SFT format as the raw data:

```bash
python scripts/convert_adapted_to_sft.py
```